In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import col, when

In [0]:
%sql
SHOW TABLES IN f1_bronze

In [0]:
df = spark.table('f1_bronze.constructor_results')
df.printSchema()

In [0]:
database_name = "f1_silver"

In [0]:
COLUMNS_GET = [
    "circuitId",
    "circuitRef",
    "name",
    "location",
    "country",
    "lat",
    "lng",
    "alt",
]
df_circuits_silver = (
    spark.table("f1_bronze.circuits")
    .select(*COLUMNS_GET)
    .withColumnRenamed("circuitId", "circuit_id")
    .withColumnRenamed("name", "circuit_name")
    .withColumnRenamed("circuitRef", "circuit_ref")
    .where("circuit_id > 10")
)
df_circuits_silver.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.circuits")

In [0]:
COLUMNS_GET = [
    "constructorResultsId", "raceId", "constructorId", "points", "status"
]
df_results_silver = (
    spark.table("f1_bronze.constructor_results")
    .select(*COLUMNS_GET)
    .withColumnRenamed("constructorResultsId", "constructor_results_id")
    .withColumnRenamed("raceId", "race_id")
    .withColumnRenamed("constructorId", "constructor_id")
    .withColumn("prata_ingestion", current_timestamp())
    .where("constructor_id > 10")
)
df_circuits_silver.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.results")

In [0]:
df_drivers_silver = (
    spark.table("f1_bronze.drivers")
    .withColumn("prata_ingestion", current_timestamp())
)
df_drivers_silver.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.drivers")

In [0]:
df_results_silver = (
    spark.table("f1_bronze.results")
    .withColumn("prata_ingestion", current_timestamp())
)
df_results_silver.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.results")


In [0]:
df_constructors_silver = (
    spark.table("f1_bronze.constructors")
    .withColumn("prata_ingestion", current_timestamp())
)
df_constructors_silver.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.constructors")

In [0]:
df_races = (
    spark.table("f1_bronze.races")
    .withColumn("prata_ingestion", current_timestamp())
)
df_races.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.races")

#### RACE_RESULTS

In [0]:
df_race_results = (
    spark.table("f1_silver.results").alias("results")
    .join(
        spark.table("f1_silver.races").alias("races"),
        on=col("results.race_id") == col("races.raceId"),
        how="inner"
    )
    .join(
        spark.table("f1_silver.circuits").alias("circuits"),
        on=col("races.circuitId") == col("circuits.circuit_id"),
        how="inner"
    )
    .join(
        spark.table("f1_silver.drivers").alias("drivers"),
        on=col("results.driver_id") == col("drivers.driverId"),
        how="inner"
    )
    .withColumn(
        "fastestLapTime",
        when(col("fastestLapTime") == "\\N", None).otherwise(col("fastestLapTime"))
    )
    .select(
        "results.race_id",
        "races.year",
        "fastestLapTime",
        "drivers.driverId",
        col("drivers.driverRef").alias("driver_name"),
        "results.position"
    )
)
df_race_results.write.format("delta").mode("overwrite").option(
    "mergeSchema", True
).saveAsTable(f"{database_name}.race_results")